# MAGIC classification using two SING density models

Fit SING separately to the **gamma (`g`)** and **hadron (`h`)** training observations, then classify held-out observations using the two fitted joint densities. This follows the Bayes-classification evaluation in [the GTM MAGIC example](../demos/demo_magic_classification.ipynb): gamma is the positive class, class priors are **0.5 / 0.5**, and results include **ROC-AUC** and **TPR at FPR 0.01, 0.02, 0.05, 0.10, 0.20**.

The notebook uses the **exact same training, validation, and test row indices** as the GTM example. Its only input preprocessing is centering and standardization with **pooled training-set column means and standard deviations**. Variables keep their original order. The original observations determine the train/validation/test split before scaling.

Use **Python (GTM SING)** in `.venv-sing`. This notebook is prepared **without execution**: the two SING fits and their derivative calculations can be expensive. `ORDER=2` is the default; model settings are fixed before evaluating the test set. SING runs on CPU, using the pinned upstream implementation and compatibility adaptations in `sing_helpers.py`.

In [ ]:
import os
from pathlib import Path
import sys
import json
import hashlib
import random
import time
import importlib.metadata as metadata

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'sing_experiments/sing_helpers.py').exists())
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.jupyter-sing/matplotlib'))
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit
from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score,
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, brier_score_loss,
    confusion_matrix, ConfusionMatrixDisplay,
)
from IPython.display import display
from sing_experiments.sing_helpers import load_sing, FixedOrdering, SING_COMMIT

print('Python:', sys.executable)
print('SING commit:', SING_COMMIT)
PACKAGE_VERSIONS = {p: metadata.version(p) for p in ['TransportMaps', 'numpy', 'scipy', 'scikit-learn']}
print(PACKAGE_VERSIONS)


## Configuration

`ORDER` is the SING polynomial degree, and `DELTA` controls its delta-method graph threshold. Both class models use the same settings and the original feature order. Training uses every row in its designated fitting fold. Validation rows are retained for density diagnostics; this notebook does not tune settings or select thresholds on the test set.

The priors match the GTM classification example. The binary decision assigns gamma when the posterior gamma probability is at least 0.5. This classification rule is separate from SING's threshold for recovering a conditional-independence graph.

In [ ]:
DATA_PATH = ROOT / 'demos/data/magic04.csv'
SPLIT_INDICES_PATH = ROOT / 'sing_experiments/magic_gtm_split_indices.npz'
FEATURE_NAMES = [f'y_{i}' for i in range(10)]
GROUPS = ('g', 'h')
SPLITS = ('train', 'validate', 'test')

ORDER = 2
DELTA = 1.0
FIT_SEED = 25
BATCH_SIZE = 16
OPTIMIZER_DERIVATIVES = 2  # Newton-CG; 1 selects BFGS
OPTIMIZER_MAXITER = 150
LOGPDF_BATCH_SIZE = 512
CLASS_PRIORS = {'g': 0.5, 'h': 0.5}
FPR_LEVELS = np.array([0.01, 0.02, 0.05, 0.10, 0.20])

assert isinstance(ORDER, int) and not isinstance(ORDER, bool) and ORDER >= 1
assert DELTA > 0 and BATCH_SIZE >= 1 and LOGPDF_BATCH_SIZE >= 1
assert all(CLASS_PRIORS[group] > 0 for group in GROUPS)
assert np.isclose(sum(CLASS_PRIORS.values()), 1.0)
config = dict(order=ORDER, delta=DELTA, fit_seed=FIT_SEED, batch_size=BATCH_SIZE,
              optimizer_derivatives=OPTIMIZER_DERIVATIVES, optimizer_maxiter=OPTIMIZER_MAXITER,
              logpdf_batch_size=LOGPDF_BATCH_SIZE, class_priors=CLASS_PRIORS,
              standardization='pooled fitting-fold mean/std, ddof=0',
              nonparanormal_transform=False, shuffle_variables=False,
              variable_order=FEATURE_NAMES, device='cpu')
config


## Load the GTM-compatible splits and standardize

The GTM helper takes the **first ceil(2/3 of each class)** as its training/validation pool and holds out the remainder for testing. It uses five Torch-randomized folds, validation fold 4, with seed 25. `magic_gtm_split_indices.npz` stores those exact row indices, generated using that same `random_split` procedure. The SING kernel therefore needs no Torch dependency to reproduce the split. A SHA-256 check ensures the indices refer to the unchanged source CSV.

Only the actual fitting rows from both classes contribute to the shared mean and standard deviation. Validation and test rows receive this same affine transformation. The class densities consequently share a coordinate system. Each original-scale log density subtracts `sum(log(training_std))`; the common correction cancels in the posterior ratio.

In [ ]:
dataframe = pd.read_csv(DATA_PATH, names=FEATURE_NAMES + ['group'])
assert dataframe['group'].isin(GROUPS).all()
raw_values = dataframe[FEATURE_NAMES].to_numpy(dtype=np.float64)
assert raw_values.shape[1] == 10 and np.isfinite(raw_values).all()
data_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

with np.load(SPLIT_INDICES_PATH, allow_pickle=False) as archive:
    split_metadata = json.loads(archive['metadata_json'].item())
    split_rows = {group: {split: archive[f'{group}_{split}'].copy() for split in SPLITS}
                  for group in GROUPS}
assert data_sha256 == split_metadata['sha256'], 'Source CSV changed: regenerate the GTM-compatible split indices'
assert split_metadata['train_portion'] == 2 / 3
assert split_metadata['cross_validation_folds'] == 5
assert split_metadata['validation_fold_index'] == 4
assert split_metadata['split_random_state'] == 25

for group in GROUPS:
    original_rows = np.flatnonzero(dataframe['group'].to_numpy() == group)
    used_rows = np.concatenate([split_rows[group][split] for split in SPLITS])
    assert len(used_rows) == len(np.unique(used_rows))
    np.testing.assert_array_equal(np.sort(used_rows), original_rows)

raw = {group: {split: raw_values[split_rows[group][split]] for split in SPLITS} for group in GROUPS}
pooled_training = np.vstack([raw[group]['train'] for group in GROUPS])
training_mean = pooled_training.mean(axis=0)
training_std = pooled_training.std(axis=0, ddof=0)
assert np.isfinite(training_std).all() and np.all(training_std > 0)
standardized = {group: {split: (raw[group][split] - training_mean) / training_std
                        for split in SPLITS} for group in GROUPS}
pooled_standardized = np.vstack([standardized[group]['train'] for group in GROUPS])
np.testing.assert_allclose(pooled_standardized.mean(axis=0), 0, atol=1e-12)
np.testing.assert_allclose(pooled_standardized.std(axis=0), 1, atol=1e-12)

split_counts = pd.DataFrame([
    dict(group=group, split=split, n=len(split_rows[group][split]))
    for group in GROUPS for split in SPLITS
]).pivot(index='group', columns='split', values='n').reindex(columns=SPLITS)
feature_scaling = pd.DataFrame({'training_mean': training_mean, 'training_std': training_std}, index=FEATURE_NAMES)
display(split_counts)
display(feature_scaling)


## Fit the two class-conditional SING models

This is the expensive step. The final fitted transport-map density is retained for classification; the recovered graph alone is insufficient to compute class likelihoods. Optimizer convergence is checked by the shared SING helper. Progress messages report derivative-batch ranges. The two class models are fitted independently, using seeds `FIT_SEED` and `FIT_SEED + 1`.

In [ ]:
models = {}
fit_rows = []
for group_index, group in enumerate(GROUPS):
    class_seed = FIT_SEED + group_index
    np.random.seed(class_seed)
    random.seed(class_seed)
    print(f'Fitting class {group}: {len(standardized[group]["train"])} observations', flush=True)
    sing, history = load_sing(
        batch_size=BATCH_SIZE, optimizer_derivatives=OPTIMIZER_DERIVATIVES,
        optimizer_maxiter=OPTIMIZER_MAXITER, retain_density=True, progress=True,
    )
    started = time.perf_counter()
    recovered_omega = sing(standardized[group]['train'].copy(), ORDER, FixedOrdering(), DELTA)
    elapsed = time.perf_counter() - started
    assert np.isfinite(recovered_omega).all()
    assert np.allclose(recovered_omega, recovered_omega.T)
    adjacency = recovered_omega != 0
    np.fill_diagonal(adjacency, False)
    diagnostics = [[dict(success=bool(log['success']),
                         accepted_by_gradient=bool(log['accepted_by_gradient']),
                         message=str(log['message']), iterations=int(log['nit']),
                         gradient_inf_norm=float(np.linalg.norm(log['jac'], ord=np.inf)))
                    for log in iteration['solver_logs']] for iteration in history]
    models[group] = dict(
        density=history[-1]['density'], graph=adjacency,
        thresholded_scores=recovered_omega.copy(),
        initial_scores=history[0]['omega_raw'].copy(),
        final_scores=history[-1]['omega_raw'].copy(),
        fit_seed=class_seed, solver_diagnostics=diagnostics,
    )
    fit_rows.append(dict(group=group, n_train=len(standardized[group]['train']),
                         iterations=len(history), edges=int(adjacency.sum() // 2),
                         elapsed_seconds=elapsed, fit_seed=class_seed))
    del history, sing
fit_summary = pd.DataFrame(fit_rows).set_index('group')
display(fit_summary)


## Held-out densities and Bayes classification

For an observation x, compute the log odds

`log_odds_gamma = log(phi_g) + log p_g(x) - log(phi_h) - log p_h(x)`.

The gamma posterior is `expit(log_odds_gamma)`. ROC-AUC and the ROC curve use the log odds, whose ranking is mathematically equivalent to the posterior ranking and avoids saturation/underflow. Gamma is encoded as 1 and hadron as 0, matching the GTM example. Both class densities are evaluated at **every** test observation.

In [ ]:
log_standardization_jacobian = float(np.log(training_std).sum())

def original_scale_logpdf(group, standardized_points):
    density = models[group]['density']
    values = np.concatenate([
        np.asarray(density.log_pdf(standardized_points[start:start + LOGPDF_BATCH_SIZE])).reshape(-1)
        for start in range(0, len(standardized_points), LOGPDF_BATCH_SIZE)
    ]) - log_standardization_jacobian
    if len(values) != len(standardized_points) or not np.isfinite(values).all():
        raise FloatingPointError(f'Nonfinite or incorrectly shaped class-{group} log densities')
    return values

# Correct-class density diagnostics; no validation or test point enters fitting.
density_rows = []
for group in GROUPS:
    for split in SPLITS:
        values = original_scale_logpdf(group, standardized[group][split])
        density_rows.append(dict(group=group, split=split, n=len(values),
                                 loglik_mean=float(values.mean()), loglik_sum=float(values.sum())))
density_loglik = pd.DataFrame(density_rows)
display(density_loglik)

test_points = np.vstack([standardized[group]['test'] for group in GROUPS])
test_row_indices = np.concatenate([split_rows[group]['test'] for group in GROUPS])
y_true = np.concatenate([np.ones(len(split_rows['g']['test']), dtype=int),
                         np.zeros(len(split_rows['h']['test']), dtype=int)])
logpdf_gamma = original_scale_logpdf('g', test_points)
logpdf_hadron = original_scale_logpdf('h', test_points)
log_odds_gamma = (np.log(CLASS_PRIORS['g']) + logpdf_gamma
                  - np.log(CLASS_PRIORS['h']) - logpdf_hadron)
assert np.isfinite(log_odds_gamma).all()
probability_gamma = expit(log_odds_gamma)
y_pred = (log_odds_gamma >= 0).astype(int)
assert len(y_true) == len(test_points) == len(test_row_indices)


## Classification scores

The TPR table follows the GTM example exactly: at each requested false-positive rate, report the largest TPR among ROC points with FPR less than or equal to that value, without interpolation. Binary metrics below use the fixed posterior-0.5 decision rule. These are **class-prediction metrics**, not measures of recovery of the unknown MAGIC conditional-independence graph.

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_true, log_odds_gamma, pos_label=1)
roc_table = pd.DataFrame({'fpr': fpr, 'tpr': tpr, 'log_odds_threshold': roc_thresholds})
tpr_at_fpr = pd.DataFrame({
    'FPR': FPR_LEVELS,
    'SING': [float(tpr[fpr <= level].max()) for level in FPR_LEVELS],
})
classification_metrics = dict(
    roc_auc=float(roc_auc_score(y_true, log_odds_gamma)),
    average_precision=float(average_precision_score(y_true, log_odds_gamma)),
    accuracy=float(accuracy_score(y_true, y_pred)),
    balanced_accuracy=float(balanced_accuracy_score(y_true, y_pred)),
    precision=float(precision_score(y_true, y_pred, zero_division=0)),
    recall=float(recall_score(y_true, y_pred, zero_division=0)),
    f1=float(f1_score(y_true, y_pred, zero_division=0)),
    mcc=float(matthews_corrcoef(y_true, y_pred)),
    binary_log_loss=float(np.logaddexp(0, np.where(y_true == 1, -log_odds_gamma, log_odds_gamma)).mean()),
    brier_score=float(brier_score_loss(y_true, probability_gamma)),
)
classification_scores = pd.DataFrame([classification_metrics], index=['SING'])
display(classification_scores)
display(tpr_at_fpr)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].plot(fpr, tpr, label=f'SING degree {ORDER}, AUC={classification_metrics["roc_auc"]:.3f}')
axes[0].plot([0, 1], [0, 1], '--', color='#94a3b8', linewidth=1)
axes[0].set(xlabel='False-positive rate (hadron classified as gamma)',
            ylabel='True-positive rate (gamma detected)', title='MAGIC test ROC',
            xlim=(0, 1), ylim=(0, 1.01))
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.2)
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=[0, 1]),
                       display_labels=['Hadron (h)', 'Gamma (g)']).plot(
    ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Fixed posterior threshold: 0.5')
plt.show()


## Save results

When executed, the following cell saves the class predictions and likelihoods, ROC points, GTM-style TPR table, summary metrics, training-only scaling, exact split indices, fit diagnostics, and class-specific SING graph/score matrices. All likelihoods use the original data scale. The dataset checksum, fixed equal priors, fitting seeds, versions, and SING commit are recorded with the results.

These files contain evaluated results and fitted-map coefficients; the notebook retains the fitted density objects in memory. Re-executing the fitting cell trains both models again.

In [ ]:
output = ROOT / 'sing_experiments/results/magic_classification' / f'p{ORDER}_delta{DELTA:g}_seed{FIT_SEED}_{data_sha256[:12]}'
output.mkdir(parents=True, exist_ok=True)
predictions = pd.DataFrame({
    'source_row_0based': test_row_indices,
    'true_group': np.where(y_true == 1, 'g', 'h'),
    'gamma_label': y_true, 'predicted_gamma_label': y_pred,
    'posterior_gamma': probability_gamma, 'log_odds_gamma': log_odds_gamma,
    'logpdf_gamma': logpdf_gamma, 'logpdf_hadron': logpdf_hadron,
})
predictions.to_csv(output / 'test_predictions.csv', index=False)
roc_table.to_csv(output / 'roc_curve.csv', index=False)
tpr_at_fpr.to_csv(output / 'tpr_at_fpr.csv', index=False)
classification_scores.to_csv(output / 'classification_metrics.csv')
density_loglik.to_csv(output / 'density_loglik.csv', index=False)
feature_scaling.to_csv(output / 'feature_scaling.csv', index_label='feature')
fit_summary.to_csv(output / 'fit_summary.csv')
arrays = dict(training_mean=training_mean, training_std=training_std,
              test_row_indices=test_row_indices, y_true=y_true, y_pred=y_pred,
              posterior_gamma=probability_gamma, log_odds_gamma=log_odds_gamma,
              logpdf_gamma=logpdf_gamma, logpdf_hadron=logpdf_hadron)
for group in GROUPS:
    for split in SPLITS:
        arrays[f'{group}_{split}_row_indices'] = split_rows[group][split]
    arrays[f'{group}_graph'] = models[group]['graph']
    arrays[f'{group}_initial_scores'] = models[group]['initial_scores']
    arrays[f'{group}_final_scores'] = models[group]['final_scores']
    arrays[f'{group}_thresholded_scores'] = models[group]['thresholded_scores']
    arrays[f'{group}_map_coefficients'] = models[group]['density'].transport_map.coeffs
np.savez_compressed(output / 'arrays.npz', **arrays)
summary = dict(config=config, split_metadata=split_metadata, sing_commit=SING_COMMIT,
               package_versions=PACKAGE_VERSIONS, classification_metrics=classification_metrics,
               loglik_scale='original observations', n_test=int(len(y_true)),
               fit_summary=fit_summary.reset_index().to_dict(orient='records'),
               solver_diagnostics={group: models[group]['solver_diagnostics'] for group in GROUPS})
(output / 'summary.json').write_text(json.dumps(summary, indent=2, allow_nan=False))
print('Saved MAGIC classification results to:', output)
